In [ ]:
import json
from pathlib import Path
import kagglehub
from collections import Counter
from email.utils import parsedate_to_datetime
import pandas as pd

In [ ]:
# Inspect the file without loading the full dataset
with file_path.open("r", encoding="utf-8") as f:
    first_record = json.loads(next(f))

print("File location:", file_path)
print("File size (GB):", round(file_path.stat().st_size / 1e9, 2))
print("Available fields:", list(first_record.keys()))
print("Category example:", first_record.get("categories"))
print("Version example:", first_record.get("versions", [])[:1])

In [ ]:
output_path = Path("data/arxiv_cscl_2021_2025.jsonl")
output_path.parent.mkdir(parents=True, exist_ok=True)

counts = Counter()
year_counts = Counter()
seen_ids = set()

with file_path.open("r", encoding="utf-8") as source, \
     output_path.open("w", encoding="utf-8") as target:

    for line in source:
        counts["total_records"] += 1
        paper = json.loads(line)

        # Include primary and cross-listed cs.CL papers
        categories = (paper.get("categories") or "").split()
        if "cs.CL" not in categories:
            continue

        counts["cscl_records"] += 1

        # Find the original submission (version v1)
        versions = paper.get("versions") or []
        v1 = next(
            (v for v in versions if v.get("version") == "v1"),
            None
        )

        if not v1 or not v1.get("created"):
            counts["missing_v1_date"] += 1
            continue

        try:
            submission_date = parsedate_to_datetime(
                v1["created"]
            ).date()
        except (TypeError, ValueError, IndexError):
            counts["invalid_v1_date"] += 1
            continue

        if not 2021 <= submission_date.year <= 2025:
            continue

        counts["candidate_records"] += 1
        year_counts[submission_date.year] += 1

        # Record potential data-quality issues
        if not str(paper.get("title") or "").strip():
            counts["missing_titles"] += 1

        if not str(paper.get("abstract") or "").strip():
            counts["missing_abstracts"] += 1

        paper_id = paper.get("id")

        if not paper_id:
            counts["missing_ids"] += 1
        elif paper_id in seen_ids:
            counts["duplicate_ids"] += 1
        else:
            seen_ids.add(paper_id)

        # Preserve original metadata and add submission date
        paper["first_submission_date"] = (
            submission_date.isoformat()
        )

        target.write(json.dumps(paper) + "\n")

print("Dataset screening:")
for key, value in counts.items():
    print(f"{key}: {value:,}")

print("\nCandidate papers by year:")
for year in sorted(year_counts):
    print(f"{year}: {year_counts[year]:,}")

print("\nSaved to:", output_path)


In [ ]:
records = []

with output_path.open("r", encoding="utf-8") as f:
    for line in f:
        paper = json.loads(line)

        records.append({
            "id": paper.get("id"),
            "title": (paper.get("title") or "").strip(),
            "abstract": (paper.get("abstract") or "").strip(),
            "categories": (paper.get("categories") or "").split(),
            "first_submission_date": paper.get("first_submission_date")
        })

df = pd.DataFrame(records)

df["first_submission_date"] = pd.to_datetime(
    df["first_submission_date"], errors="coerce"
)

print("Total records:", len(df))
print("Unique IDs:", df["id"].nunique())
print("Duplicate IDs:", df["id"].duplicated().sum())
print("Missing IDs:", df["id"].fillna("").eq("").sum())
print("Empty titles:", df["title"].eq("").sum())
print("Empty abstracts:", df["abstract"].eq("").sum())
print("Invalid dates:", df["first_submission_date"].isna().sum())

print(
    "Records without cs.CL:",
    (~df["categories"].apply(lambda x: "cs.CL" in x)).sum()
)

print(
    "Records outside 2021–2025:",
    (~df["first_submission_date"].dt.year.between(2021, 2025)).sum()
)

print("\nDate range:")
print(df["first_submission_date"].min())
print(df["first_submission_date"].max())

print("\nFive randomly selected papers:")
print(
    df.sample(5, random_state=42)[
        ["first_submission_date", "title"]
    ].to_string(index=False)
)

In [ ]:

# Normalize whitespace without modifying the original columns
df["clean_title"] = (
    df["title"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

df["clean_abstract"] = (
    df["abstract"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Calculate document lengths in words
df["title_words"] = df["clean_title"].str.split().str.len()
df["abstract_words"] = df["clean_abstract"].str.split().str.len()

# Examine the distributions
print("Text length statistics:")
print(
    df[["title_words", "abstract_words"]]
    .describe(percentiles=[0.01, 0.05, 0.95, 0.99])
    .round(1)
)

# Inspect unusually short abstracts
print("\nFive shortest abstracts:")
print(
    df.nsmallest(5, "abstract_words")[
        ["id", "abstract_words", "clean_title"]
    ].to_string(index=False)
)

# Inspect unusually long abstracts
print("\nFive longest abstracts:")
print(
    df.nlargest(5, "abstract_words")[
        ["id", "abstract_words", "clean_title"]
    ].to_string(index=False)
)

In [ ]:
# Examine selected papers with unusual abstract lengths
paper_ids = [
    "2207.02657",
    "2411.07623",
    "2402.16406",
    "2503.22000",
    "2309.02812"
]

for paper_id in paper_ids:
    paper = df.loc[df["id"] == paper_id].iloc[0]

    print("=" * 70)
    print("ID:", paper["id"])
    print("Title:", paper["clean_title"])
    print("Categories:", paper["categories"])
    print("Abstract length:", paper["abstract_words"])
    print("Abstract:")
    print(paper["clean_abstract"])
    print()


In [ ]:
# Identify unusually short abstracts
short_abstracts = df[df["abstract_words"] < 40]

print("Total papers:", len(df))
print("Abstracts shorter than 40 words:", len(short_abstracts))
print(
    "Percentage:",
    round(100 * len(short_abstracts) / len(df), 2),
    "%"
)

# Check whether short abstracts are concentrated in certain years
df["year"] = df["first_submission_date"].dt.year

short_counts = (
    short_abstracts["first_submission_date"]
    .dt.year
    .value_counts()
    .sort_index()
)

yearly_summary = pd.DataFrame({
    "total_papers": df.groupby("year").size(),
    "short_abstracts": short_counts
}).fillna(0)

yearly_summary["short_percentage"] = (
    100 * yearly_summary["short_abstracts"]
    / yearly_summary["total_papers"]
).round(2)

print("\nShort abstracts by year:")
print(yearly_summary)

# Inspect a reproducible random sample of short abstracts
print("\nRandom sample of short abstracts:")

sample = short_abstracts.sample(
    n=min(8, len(short_abstracts)),
    random_state=42
)

for _, paper in sample.iterrows():
    print("\n", "=" * 60)
    print("ID:", paper["id"])
    print("Title:", paper["clean_title"])
    print("Abstract:", paper["clean_abstract"])
